In [ ]:
from netCDF4 import Dataset
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import datetime
from tqdm import tqdm
from glob import glob
import pickle
import random
import os
import fnmatch

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as TorchDataset
import torch.nn.functional as F
from torchvision import datasets, transforms
from typing import Tuple, List, Type, Dict, Any
from torch.utils.tensorboard import SummaryWriter

from SGDR import CosineAnnealingWarmRestarts
from mish import Mish
from coord_conv import CoordConv
from MyResidualNetwork import MyResNet, MyBasicBlock
from MyDataPreparation_0125 import CustomDataset, Sampler
from autoencoder import Encoder, Decoder

In [ ]:
def drawing(tensor1, tensor2, indexes):
    for i in indexes:
        extracted_tensor1_0 = tensor1[i, 0, :, :]
        extracted_tensor1_1 = tensor1[i, 1, :, :]
        extracted_tensor1_2 = tensor1[i, 2, :, :]
        extracted_tensor2_0 = tensor2[i, 0, :, :]
        extracted_tensor2_1 = tensor2[i, 1, :, :]
        extracted_tensor2_2 = tensor2[i, 2, :, :]

        array1_0 = extracted_tensor1_0.detach().cpu().numpy()
        array1_1 = extracted_tensor1_1.detach().cpu().numpy()
        array1_2 = extracted_tensor1_2.detach().cpu().numpy()
        array2_0 = extracted_tensor2_0.detach().cpu().numpy()
        array2_1 = extracted_tensor2_1.detach().cpu().numpy()
        array2_2 = extracted_tensor2_2.detach().cpu().numpy()

        vmin = min(array1_0.min(), array2_0.min())
        vmax = max(array1_0.max(), array2_0.max())

        fig, axs = plt.subplots(2, 3, figsize=(10, 5))

        cax1 = axs[0, 0].imshow(array1_0, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
        axs[0, 0].set_title('Sample adt')

        cax2 = axs[0, 1].imshow(array1_1, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
        axs[0, 1].set_title('Sample ugos')
        
        cax3 = axs[0, 2].imshow(array1_2, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
        axs[0, 2].set_title('Sample vgos')

        cax4 = axs[1, 0].imshow(array2_0, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
        axs[1, 0].set_title('Decoded adt')

        cax5 = axs[1, 1].imshow(array2_1, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
        axs[1, 1].set_title('Decoded ugos')

        cax6 = axs[1, 2].imshow(array2_2, cmap='viridis', aspect='auto', vmin=vmin, vmax=vmax)
        axs[1, 2].set_title('Decoded vgos')

        cbar = fig.colorbar(cax1, ax=axs, orientation='vertical', fraction=0.02, pad=0.04)
        cbar.ax.set_ylabel('adt')

        plt.tight_layout()
        plt.show()

        print('--------------------------------------------------------------------------------------------------')

In [ ]:
data = Dataset('/mnt/hippocamp/asavin/data/adt/adt_1993-2024_daily_n80_s70_w55_e105.nc', 'r')

dataset = CustomDataset(data=data)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
data.close()

encoder = Encoder(in_channels=3, H=80, W=400, expansions=[4, 4, 4, 4], n_blocks=26, decreases=[2, 2, 2, 2], bottleneck=32)
decoder = Decoder(in_features=encoder.bottleneck, start_channels=768, finish_channels=encoder.in_channels, n_layers=4,
                  expansion_value=0.25, increase_value=2, H=5, W=25, H_out=80, W_out=400)

In [ ]:
encoder = encoder.cuda()
decoder = decoder.cuda()

In [ ]:
data, land_mask, _ = next(iter(dataloader))
data.shape, land_mask.shape

In [ ]:
data_gpu = data.to(device='cuda', dtype=torch.float)
land_mask_gpu = land_mask.to(device='cuda', dtype=torch.float)

encoded_data = encoder.forward(data_gpu)
decoded_data = decoder.forward(encoded_data)

data_gpu_masked = data_gpu[land_mask_gpu == 1]
result_masked = decoded_data[land_mask_gpu == 1]

In [ ]:
loss_function=torch.nn.MSELoss()

In [ ]:
loss = loss_function(data_gpu_masked, result_masked)

In [ ]:
loss.backward()

In [ ]:
run_name = 'adt_pre_autoencoder_run004'

In [ ]:
device = torch.device('cuda:1')

In [ ]:
encoder = torch.load(f'/app/Kara_plume_movement/adt/models/model_{run_name}_encoder.pth', map_location=torch.device('cpu'));
decoder = torch.load(f'/app/Kara_plume_movement/adt/models/model_{run_name}_decoder.pth', map_location=torch.device('cpu'));

In [ ]:
encoder.eval();
decoder.eval();

In [ ]:
encoder = encoder.cuda()
decoder = decoder.cuda()

In [ ]:
from adt_pre_autoencoder_004 import validate_single_epoch, find_files

In [ ]:
data = Dataset('/mnt/hippocamp/asavin/data/adt/adt_1993-2024_daily_n80_s70_w55_e105.nc', 'r')

In [ ]:
batch_size = 4

In [ ]:
dataset = CustomDataset(data=data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
data, land_mask, _ = next(iter(dataloader))
data.shape, land_mask.shape

In [ ]:
loss_function=torch.nn.MSELoss()

In [ ]:
data_gpu = data.to(device='cuda', dtype=torch.float)
land_mask_gpu = land_mask.to(device='cuda', dtype=torch.float)
encoded_data = encoder.forward(data_gpu)
decoded_data = decoder.forward(encoded_data)
data_gpu_masked = data_gpu[land_mask_gpu == 1]
result_masked = decoded_data[land_mask_gpu == 1]
loss = loss_function(data_gpu_masked, result_masked)
test_loss = loss.detach() * batch_size

In [ ]:
test_loss

In [ ]:
data_gpu_drawing = data_gpu * land_mask_gpu
decoded_data_drawing = decoded_data * land_mask_gpu
data_gpu_drawing.shape, decoded_data_drawing.shape

In [ ]:
drawing(data_gpu_drawing, decoded_data_drawing, [i for i in range(data_gpu.shape[0])])